## What is Word Embedding?

Word embedding is a technique in Natural Language Processing (NLP) where words are converted into dense numerical vectors (lists of numbers) that capture the meaning and relationships between words.

Instead of representing words as simple IDs or sparse vectors (like one-hot encoding), embeddings place similar words closer together in vector space.

1. For example:

- "king" and "queen" → similar vectors
- "dog" and "puppy" → closer than "dog" and "car"

This helps machine learning models understand language better.

<b>Why not One-Hot Encoding?</b>
- In one-hot encoding:

<table>
    <th>Word</th>	                   <th>  Vector</th>
 <tr>      
    <td>cat  </td>                      	<td>[1,0,0,0]</td> </tr> 
<tr>      
<td>dog</td>	                       <td> [0,1,0,0]</td> </tr> 
<tr>      
<td>car	</td>                       <td> [0,0,1,0]</td> </tr> </table>



## Types of Word Embeddings
- <B>Word2Vec – Predicts neighboring words</b>
- <B>CBOW (Continuous Bag of Words)</b>
- <B>Skip-Gram</b>
- <B>GloVe – Based on word co-occurrence matrix</b>
- <B>FastText – Uses subwords/character n-grams</b>
-<B> Contextual Embeddings – Different meaning based on sentence</b>
1. BERT
2. GPT

In [1]:
from sklearn.preprocessing import OneHotEncoder

# Data
colors = [['Red'], ['Blue'], ['Green']]

# Create Encoder
encoder = OneHotEncoder(sparse_output=False)

# Transform data
encoded = encoder.fit_transform(colors)

print(encoded)

[[0. 0. 1.]
 [1. 0. 0.]
 [0. 1. 0.]]


In [2]:
pip install gensim

Note: you may need to restart the kernel to use updated packages.


In [3]:
from gensim.models import Word2Vec

# Sample Sentences
sentences = [
    ['i', 'love', 'python'],
    ['python', 'is', 'easy'],
    ['i', 'love', 'machine', 'learning'],
    ['machine', 'learning', 'is', 'powerful']
]

# Train Word2Vec model
model = Word2Vec(
    sentences,
    vector_size=5,         # embedding dimension
    window=2,              # nearby words
    min_count=1,
    workers=4
)

# Get embedding vector for a word
vector = model.wv['love']
print("Embedding for love:")
print(vector)

Embedding for love:
[0.14623532 0.10140524 0.13515386 0.01525731 0.12701781]


In [4]:
import numpy as np
import pandas as pd
import re
import nltk
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

In [5]:
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ANMOL\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ANMOL\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [7]:
df = pd.read_csv("IMDB Dataset.csv")

print(df.head())
print(df.shape)

df['sentiment'] = df['sentiment'].map({
    'positive': 1,
    'negative': 0
})

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive
(50000, 2)


In [8]:
stop_words = set(stopwords.words('english'))

def clean_text(text):
    # Lowercase
    text = text.lower()

    # Rename HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Rename special characters
    text = re.sub(r'[^a-zA-Z]', '', text)

    # Tokenization
    words = word_tokenize(text)

    # Rename stopwords
    words = [word for word in words if word not in stop_words]

    return words

# Apply preprocessing
df['clean_review'] = df['review'].apply(clean_text)

print(df[['review', 'clean_review']].head())

                                              review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   
3  Basically there's a family where a little boy ...   
4  Petter Mattei's "Love in the Time of Money" is...   

                                        clean_review  
0  [oneoftheotherreviewershasmentionedthatafterwa...  
1  [awonderfullittleproductionthefilmingtechnique...  
2  [ithoughtthiswasawonderfulwaytospendtimeonatoo...  
3  [basicallytheresafamilywherealittleboyjakethin...  
4  [pettermatteisloveinthetimeofmoneyisavisuallys...  


In [10]:
Word2Vec_model = Word2Vec(
    sentences=df['clean_review'],
    vector_size=100,
    window=5,
    min_count=2,
    workers=4
)

print("Word2Vec Vocabulary Size:",
      len(Word2Vec_model.wv))

Word2Vec Vocabulary Size: 413


In [12]:
# Convert words to text again
texts = df['clean_review'].apply(lambda x: " ".join(x))

tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)

X = tokenizer.texts_to_sequences(texts)

# Padding sequences
max_length = 100
X = pad_sequences(X, maxlen=max_length)

y = df['sentiment'].values

print("Shape of X:", X.shape)
print("Shape of y:", y.shape)

Shape of X: (50000, 100)
Shape of y: (50000,)


In [16]:
vocab_size = len(tokenizer.word_index) + 1
embedding_dim = 100

embedding_matrix = np.zeros(
    (vocab_size, embedding_dim)
)

for word, i in tokenizer.word_index.items():
    if word in Word2Vec_model.wv:
        embedding_matrix[i] = Word2Vec_model.wv[word]

print("Embedding Matrix Shape:",
      embedding_matrix.shape)

Embedding Matrix Shape: (49576, 100)


In [17]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

In [19]:
model = Sequential()

# Embedding Layer
model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=embedding_dim,
        weights=[embedding_matrix],
        input_length=max_length,
        trainable=False
    )
)

# LSTM Layer
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))

# Dense Layers
model.add(Dense(64, activation='relu'))
model.add(Dropout(0.5))

# Output Layer
model.add(Dense(1, activation='sigmoid'))

# Compile model
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
#Model Summary
model.summary()

C:\Users\ANMOL\OneDrive\Attachments\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │     4,957,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,957,600 (18.91 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 4,957,600 (18.91 MB)

In [20]:
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=64,
    validation_data=(X_test, y_test)
)

Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 134s 193ms/step - accuracy: 0.4973 - loss: 0.6933 - val_accuracy: 0.4961 - val_loss: 0.6932
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 108s 173ms/step - accuracy: 0.4965 - loss: 0.6932 - val_accuracy: 0.5085 - val_loss: 0.6931
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 121s 194ms/step - accuracy: 0.4994 - loss: 0.6932 - val_accuracy: 0.4961 - val_loss: 0.6932
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 120s 192ms/step - accuracy: 0.4938 - loss: 0.6932 - val_accuracy: 0.5039 - val_loss: 0.6931
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 121s 193ms/step - accuracy: 0.5011 - loss: 0.6932 - val_accuracy: 0.4961 - val_loss: 0.6932


In [23]:
loss, accuracy = model.evaluate(X_test, y_test)

print("\nTest Accuracy:", accuracy)

# Predictions
y_pred = model.predict(X_test)
y_pred = (y_pred > 0.5).astype(int)

# Metrics
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

313/313 ━━━━━━━━━━━━━━━━━━━━ 10s 32ms/step - accuracy: 0.4961 - loss: 0.6932

Test Accuracy: 0.4961000084877014
313/313 ━━━━━━━━━━━━━━━━━━━━ 12s 38ms/step

Classification Report:
              precision    recall  f1-score   support

           0       0.50      1.00      0.66      4961
           1       0.00      0.00      0.00      5039

    accuracy                           0.50     10000
   macro avg       0.25      0.50      0.33     10000
weighted avg       0.25      0.50      0.33     10000


Confusion Matrix:
[[4961    0]
 [5039    0]]


C:\Users\ANMOL\OneDrive\Attachments\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\ANMOL\OneDrive\Attachments\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\ANMOL\OneDrive\Attachments\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [24]:
def predict_sentiment(text):

    cleaned = clean_text(text)

    seq = tokenizer.texts_to_sequences(
        [" ".join(cleaned)]
    )

    padded = pad_sequences(
        seq,
        maxlen=max_length
    )

    prediction = model.predict(padded)[0][0]

    if prediction > 0.5:
        return "Positive Review"
    else:
        return "Negative Review"


# Test custom review
review = input("Enter review: ")

result = predict_sentiment(review)

print("Prediction:", result)

Enter review:  


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 160ms/step
Prediction: Negative Review
